# Day I19 — FOG 탐지 모델: 참가자 단위 검증과 event 단위 평가

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I19.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I19.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — window feature 만들기와 threshold baseline(전역 LOSO vs 환자별)

In [ ]:
import glob, os
import numpy as np
import pandas as pd

DATA = "data/daphnet/dataset_fog_release/dataset"
OUT = "results/advanced/I"
os.makedirs(OUT, exist_ok=True)
fs, WIN, STEP = 64, 256, 32
SENSORS = {"shank": 2, "thigh": 5, "hip": 8}          # 각 센서의 수직축 열 번호(0부터)
f = np.fft.rfftfreq(WIN, 1 / fs)
LOCO, FRZ = (f >= 0.5) & (f < 3), (f >= 3) & (f <= 8)

def window_features(a):
    ends = np.arange(WIN - 1, len(a), STEP)
    idx = ends[:, None] - np.arange(WIN)[::-1]
    feat = {"t_end": ends / fs, "label": a[ends, 10]}
    for name, col in SENSORS.items():
        seg = a[idx, col]
        seg = seg - seg.mean(axis=1, keepdims=True)
        p = np.abs(np.fft.rfft(seg, axis=1)) ** 2 / WIN
        loco, frz = p[:, LOCO].sum(1), p[:, FRZ].sum(1)
        feat[f"{name}_energy"] = loco + frz
        feat[f"{name}_logFI"] = np.log10((frz + 1) / (loco + 1))   # +1: 0으로 나누기 방지
        feat[f"{name}_logE"] = np.log10(loco + frz + 1)
        feat[f"{name}_sd"] = seg.std(axis=1)
    return pd.DataFrame(feat)

parts = []
for path in sorted(glob.glob(os.path.join(DATA, "S*R*.txt"))):
    a = np.loadtxt(path)
    w = window_features(a)
    w.insert(0, "run", os.path.basename(path)[3:6])
    w.insert(0, "subject", os.path.basename(path)[:3])
    parts.append(w)
win = pd.concat(parts, ignore_index=True)
win = win[win["label"] > 0].reset_index(drop=True)
win["fog"] = (win["label"] == 2).astype(int)
win.to_csv(os.path.join(OUT, "I19_windows.csv.gz"), index=False)
print(win.shape, "| FOG window 비율 %.1f%%" % (100 * win["fog"].mean()))

# ---- 1단계: freeze index threshold baseline (shank 수직축) ----
FI = 10 ** win["shank_logFI"].to_numpy()
E = win["shank_energy"].to_numpy()
y = win["fog"].to_numpy().astype(bool)
subj = win["subject"].to_numpy()
FTH = np.round(np.logspace(-0.3, 1.3, 33), 2)        # FreezeTH 후보 0.5–20
PTH = [1e3, 1e4, 1e5, 1e6]                           # PowerTH 후보

def sens_spec(pred, yy):
    se = pred[yy].mean() if yy.any() else np.nan
    sp = (~pred[~yy]).mean()
    return se, sp

def score(fth, pth, mask):
    """mask 안 참가자마다 min(민감도, 특이도)를 구해 평균(FOG 없는 참가자는 제외)."""
    vals = []
    for s in np.unique(subj[mask]):
        m = mask & (subj == s)
        if not y[m].any():
            continue
        se, sp = sens_spec((FI[m] > fth) & (E[m] >= pth), y[m])
        vals.append(min(se, sp))
    return np.mean(vals)

rows = []
for s in np.unique(subj):
    test = subj == s
    # (a) 전역 threshold: 나머지 9명으로 고르고 s에 적용 (LOSO)
    g = max(((score(ft, pt, ~test), ft, pt) for ft in FTH for pt in PTH))
    se_g, sp_g = sens_spec((FI[test] > g[1]) & (E[test] >= g[2]), y[test])
    row = {"subject": s, "glob_FTH": g[1], "glob_PTH": g[2],
           "glob_sens": se_g, "glob_spec": sp_g}
    # (b) 환자별 threshold: s 자신의 자료로 고른 값(낙관적 상한)
    if y[test].any():
        b = max(((score(ft, pt, test), ft, pt) for ft in FTH for pt in PTH))
        se_b, sp_b = sens_spec((FI[test] > b[1]) & (E[test] >= b[2]), y[test])
        row.update({"own_FTH": b[1], "own_PTH": b[2], "own_sens": se_b, "own_spec": sp_b})
    rows.append(row)
res = pd.DataFrame(rows)
pct = ["glob_sens", "glob_spec", "own_sens", "own_spec"]
res[pct] = (100 * res[pct]).round(1)
print(res.to_string(index=False, na_rep="-"))
res.to_csv(os.path.join(OUT, "I19_thresholds.csv"), index=False)
fog = res.dropna(subset=["own_sens"])
print("FOG 있는 8명 평균 | 전역: 민감도 %.1f, 특이도 %.1f | 환자별: 민감도 %.1f, 특이도 %.1f" % (
    fog.glob_sens.mean(), fog.glob_spec.mean(), fog.own_sens.mean(), fog.own_spec.mean()))
print("FOG 있는 8명 평균 min(민감도, 특이도) | 전역 %.1f | 환자별 %.1f" % (
    np.minimum(fog.glob_sens, fog.glob_spec).mean(), np.minimum(fog.own_sens, fog.own_spec).mean()))

### 예시 2 — LOSO로 logistic regression과 random forest 평가하기

예시 1이 저장한 `results/advanced/I/I19_windows.csv.gz`를 읽습니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

win = pd.read_csv("results/advanced/I/I19_windows.csv.gz")
FEATS = [c for c in win.columns if c.endswith(("_logFI", "_logE", "_sd"))]   # 3 센서 × 3 = 9개
X, y, groups = win[FEATS].to_numpy(), win["fog"].to_numpy(), win["subject"].to_numpy()
print("feature:", FEATS)

models = {
    # scaler는 pipeline 안에 → 각 fold의 train 9명으로만 평균·SD를 맞춘다(누수 방지)
    "LR": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)),
    "RF": RandomForestClassifier(n_estimators=200, min_samples_leaf=20, class_weight="balanced_subsample",
                                 n_jobs=-1, random_state=0),
}
for name, model in models.items():
    prob = np.zeros(len(y))
    for tr, te in LeaveOneGroupOut().split(X, y, groups):       # 한 번에 한 사람을 통째로 test
        model.fit(X[tr], y[tr])
        prob[te] = model.predict_proba(X[te])[:, 1]
    win[f"p_{name}"] = prob
win.to_csv("results/advanced/I/I19_loso_pred.csv.gz", index=False)

rows = []
for s, g in win.groupby("subject"):
    yy = g["fog"].to_numpy().astype(bool)
    row = {"subject": s, "FOG_windows": int(yy.sum())}
    for name in models:
        pred = g[f"p_{name}"].to_numpy() >= 0.5
        row[f"{name}_sens"] = round(100 * pred[yy].mean(), 1) if yy.any() else np.nan
        row[f"{name}_spec"] = round(100 * (~pred[~yy]).mean(), 1)
    rows.append(row)
tab = pd.DataFrame(rows)
print(tab.to_string(index=False, na_rep="-"))
fog = tab[tab.FOG_windows > 0]
for name in models:
    print("%s: FOG 있는 8명 평균 민감도 %.1f (범위 %.1f–%.1f), 10명 평균 특이도 %.1f (범위 %.1f–%.1f)" % (
        name, fog[f"{name}_sens"].mean(), fog[f"{name}_sens"].min(), fog[f"{name}_sens"].max(),
        tab[f"{name}_spec"].mean(), tab[f"{name}_spec"].min(), tab[f"{name}_spec"].max()))

### 예시 3 — event 단위: 삽화를 놓쳤나, 언제 울렸나, 몇 번 헛울렸나

In [ ]:
import numpy as np
import pandas as pd

win = pd.read_csv("results/advanced/I/I19_loso_pred.csv.gz")
th = pd.read_csv("results/advanced/I/I19_thresholds.csv").set_index("subject")
# 전역(LOSO) FI threshold의 예측도 같은 형식으로 붙인다
fi = 10 ** win["shank_logFI"]
win["pred_FI"] = (fi > win["subject"].map(th["glob_FTH"])) & (win["shank_energy"] >= win["subject"].map(th["glob_PTH"]))
win["pred_LR"], win["pred_RF"] = win["p_LR"] >= 0.5, win["p_RF"] >= 0.5
TOL = 2.0   # 삽화가 끝난 뒤 2초까지 켜진 경보는 "그 삽화를 잡은 것"으로 인정(Bächlin의 2 s 허용과 같은 발상)

def events(g):
    """window 라벨 열에서 FOG 삽화의 [시작, 끝] 시각(window 끝 시각 기준)을 찾는다."""
    fog = g["fog"].to_numpy().astype(bool); t = g["t_end"].to_numpy()
    gap = np.r_[True, np.diff(t) > 0.51]                 # 라벨 0 구간으로 끊긴 곳
    start = fog & (~np.r_[False, fog[:-1]] | gap)
    stop = fog & (~np.r_[fog[1:], False] | np.r_[gap[1:], True])
    return list(zip(t[start], t[stop]))

rows = []
for (s, r), g in win.sort_values(["subject", "run", "t_end"]).groupby(["subject", "run"]):
    t = g["t_end"].to_numpy()
    ev = events(g)
    for m in ["FI", "LR", "RF"]:
        pred = g[f"pred_{m}"].to_numpy().astype(bool)
        for on, off in ev:
            hit = pred & (t >= on) & (t <= off + TOL)
            rows.append({"subject": s, "model": m, "kind": "event", "detected": float(hit.any()),
                         "latency": t[hit][0] - on if hit.any() else np.nan,
                         "already_on": float(pred[(t > on - 0.75) & (t < on)].any())})  # 시작 직전 window가 이미 양성
        inside = np.zeros(len(t), bool)
        for on, off in ev:
            inside |= (t >= on - TOL) & (t <= off + TOL)   # 시작 2초 전 경보도 오경보로 세지 않음
        rise = pred & ~np.r_[False, pred[:-1]]            # 경보가 새로 켜진 순간
        rows.append({"subject": s, "model": m, "kind": "alarm",
                     "false_alarms": int((rise & ~inside).sum()),
                     "nonfog_min": (g["fog"] == 0).sum() * 0.5 / 60})
d = pd.DataFrame(rows)
ev, al = d[d.kind == "event"], d[d.kind == "alarm"]
tab = ev.groupby(["model", "subject"]).agg(events=("detected", "size"), detect_pct=("detected", "mean"),
                                           latency_med_s=("latency", "median"), already_on_pct=("already_on", "mean"))
fa = al.groupby(["model", "subject"])[["false_alarms", "nonfog_min"]].sum()
tab = tab.join(fa, how="outer")
tab["FA_per_10min"] = 10 * tab["false_alarms"] / tab["nonfog_min"]
tab["detect_pct"] *= 100
tab["already_on_pct"] *= 100
cols = ["events", "detect_pct", "latency_med_s", "already_on_pct", "FA_per_10min"]
print(tab.loc["RF", cols].round(1).to_string(na_rep="-"))     # 참가자별 표는 RF만 출력(FI, LR도 같은 방식)
for m in ["FI", "LR", "RF"]:
    e = ev[ev.model == m]; a = al[al.model == m]
    print("%s: 삽화 %d개 중 %.1f%% 검출 | 지연 중앙값 %.1f s (IQR %.1f–%.1f) | 시작 전에 이미 켜짐 %.1f%% | 오경보 %.1f회/비FOG 10분" % (
        m, len(e), 100 * e.detected.mean(), e.latency.median(), e.latency.quantile(.25), e.latency.quantile(.75),
        100 * e.already_on.mean(), 10 * a.false_alarms.sum() / a.nonfog_min.sum()))